# Iterators and Generators

In chapter 10 we read a large file one line at a time and said it was "memory-efficient". In chapter 5 we wrote `for item in my_list` hundreds of times. In chapter 7 we used `enumerate()` and `zip()`.

All three were the same machinery, and we never looked at it.

This chapter opens it up. The payoff is practical: once you understand it you can work with data that is **larger than your computer's memory**, which is a routine situation in data science and the reason this topic matters before you touch a real dataset.

**What we will learn:**

1. What a `for` loop actually does, step by step
2. The difference between an *iterable* and an *iterator*
3. Writing your own iterator with `__iter__` and `__next__`
4. Generators: the same thing in a fraction of the code
5. Generator expressions, and the rest of the comprehension family
6. Reading large files lazily
7. When a generator is the wrong choice

---
# 1. What a `for` Loop Actually Does

Start with something you have written a hundred times:

In [1]:
shows = ["Stranger Things", "Dark", "Wednesday"]

for show in shows:
    print(show)

Stranger Things
Dark
Wednesday


Python does not "look inside the list" and it does not use an index counter. What it actually does is:

1. Ask the list for an **iterator** — an object whose job is to walk through it.
2. Ask that iterator for the next item, over and over.
3. Stop when the iterator says there is nothing left.

You can do all three by hand. `iter()` performs step 1 and `next()` performs step 2:

In [2]:
it = iter(shows)

print(next(it))
print(next(it))
print(next(it))

Stranger Things
Dark
Wednesday


Three items, three calls. So what happens on a fourth?

In [3]:
try:
    next(it)
except StopIteration:
    print("StopIteration: the iterator is exhausted")

StopIteration: the iterator is exhausted


That is the whole mechanism. An iterator signals "I am finished" by **raising an exception** — `StopIteration`, one of the exception types you met in chapter 9.

A `for` loop is just this pattern with the exception handling written for you. These two cells do exactly the same work:

In [4]:
# The for loop you write
for show in shows:
    print(show)

Stranger Things
Dark
Wednesday


In [5]:
# What Python effectively runs
it = iter(shows)
while True:
    try:
        show = next(it)
    except StopIteration:
        break
    print(show)

Stranger Things
Dark
Wednesday


Nobody writes the second version. But knowing it is there explains a lot of behaviour that otherwise looks arbitrary — including several bugs later in this chapter.

## Almost Everything Is Iterable

Anything you can put after `in` in a `for` loop is **iterable**. That is a much wider set of things than lists:

In [6]:
print(list(iter("Dark")))                       # strings, character by character
print(list(iter((1, 2, 3))))                    # tuples
print(list(iter({"a": 1, "b": 2})))             # dicts -> the keys
print(sorted(iter({"x", "y"})))                 # sets -- sorted, because a set has no fixed order
print(list(iter(range(4))))                     # range objects

['D', 'a', 'r', 'k']
[1, 2, 3]
['a', 'b']
['x', 'y']
[0, 1, 2, 3]


`range` is worth a second look, because it catches people out. `range(1000000)` does **not** build a list of a million numbers:

In [7]:
r = range(1_000_000)

print(r)                    # it prints as a range, not as [0, 1, 2, ...]
print(type(r))
print(r[500])               # you can still index it
print(sum(r))               # and consume it -- the numbers are produced on demand

range(0, 1000000)
<class 'range'>
500
499999500000


It stores three numbers — start, stop and step — and produces values as you ask for them. That idea, **producing values on demand instead of storing them all**, is the whole subject of this chapter. `range` is simply the version you have been using since chapter 5 without noticing.

`enumerate()` and `zip()` from chapter 7 work the same way:

In [8]:
e = enumerate(["a", "b"])

print(type(e))              # not a list
print(list(e))              # it only becomes one when you ask

<class 'enumerate'>
[(0, 'a'), (1, 'b')]


---
## Iterable vs Iterator

Two words that sound the same and are not:

| | Iterable | Iterator |
|---|---|---|
| What it is | Something you *can* loop over | The thing that *does* the looping |
| Special method | `__iter__` | `__next__` (and `__iter__`) |
| Examples | list, string, dict, set, file | what `iter(my_list)` gives you |
| Reusable? | **Yes** — loop over it as often as you like | **No** — once exhausted, it is finished |

That last row is the one that causes bugs. A list can be looped over repeatedly:

In [9]:
numbers = [1, 2, 3]

print(sum(numbers))
print(sum(numbers))         # same answer, the list is unchanged

6
6


An iterator cannot. It has no way to go back:

In [10]:
it = iter(numbers)

print(sum(it))
print(sum(it))              # already exhausted -- there is nothing left to add

6
0


**Zero, not three.** Nothing raised an error and nothing warned us; the second `sum()` simply found an empty iterator.

Remember this. It is the single most common generator bug, and we will meet it again in a more realistic form later.

One more distinction worth seeing directly. Calling `iter()` on a **list** creates a *new* iterator every time, but calling `iter()` on an **iterator** just hands the same object back:

In [11]:
numbers = [1, 2, 3]

print(iter(numbers) is iter(numbers))     # two separate iterators over one list

it = iter(numbers)
print(iter(it) is it)                     # an iterator is its own iterator

False
True


That second line is why you can pass an iterator directly to a `for` loop: the loop calls `iter()` on whatever it is given, and an iterator answers "that is already me".

---
# 2. Writing Your Own Iterator

To make your own class work in a `for` loop, give it the two methods a `for` loop calls: `__iter__`, which returns the iterator, and `__next__`, which produces the next value or raises `StopIteration`.

These are dunder methods, the same family as `__init__` and `__str__` from chapter 8.

Here is a countdown, written the long way:

In [12]:
class Countdown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self                     # this object is its own iterator

    def __next__(self):
        if self.current <= 0:
            raise StopIteration         # tell the loop we are done
        value = self.current
        self.current -= 1
        return value

In [13]:
for n in Countdown(5):
    print(n, end=" ")

5 4 3 2 1 

It works, and it works everywhere a `for` loop works — including in `list()`, `sum()` and `max()`, since all of them use the same protocol:

In [14]:
print(list(Countdown(4)))
print(sum(Countdown(4)))
print(max(Countdown(4)))

[4, 3, 2, 1]
10
4


Here is something more realistic: paging through results the way an API makes you. You ask for a page at a time, and you do not know in advance how many there are.

In [15]:
class SearchResults:
    # Walks every result across every page, fetching one page at a time.
    def __init__(self, pages):
        self.pages = pages
        self.page_index = 0
        self.item_index = 0

    def __iter__(self):
        return self

    def __next__(self):
        while self.page_index < len(self.pages):
            page = self.pages[self.page_index]
            if self.item_index < len(page):
                item = page[self.item_index]
                self.item_index += 1
                return item
            self.page_index += 1        # this page is done, move to the next
            self.item_index = 0
        raise StopIteration

In [16]:
pages = [["Dune", "Arrival"], ["Sicario"], ["Blade Runner 2049", "Prisoners"]]

for title in SearchResults(pages):
    print(title)

Dune
Arrival
Sicario
Blade Runner 2049
Prisoners


That is fifteen lines of bookkeeping — two index variables, a `while`, and a manual `StopIteration` — to express a very simple idea: *give me each item from each page in turn*.

Generators exist because that ratio is unacceptable.

---
# 3. Generators: The Same Thing, Far Less Code

A **generator** is a function that produces a series of values instead of returning one. You write it exactly like a normal function, with one change: use **`yield`** instead of `return`.

Here is the entire `SearchResults` class from above, rewritten:

In [17]:
def search_results(pages):
    for page in pages:
        for item in page:
            yield item

In [18]:
for title in search_results(pages):
    print(title)

Dune
Arrival
Sicario
Blade Runner 2049
Prisoners


Four lines instead of fifteen, no index variables, no manual `StopIteration`, and the logic now reads as the plain-English sentence it always was.

That inner loop — *"yield every item from this thing"* — is common enough to have its own syntax, **`yield from`**:

In [19]:
def search_results(pages):
    for page in pages:
        yield from page         # hand over every item in page, one at a time

In [20]:
print(list(search_results(pages)))

['Dune', 'Arrival', 'Sicario', 'Blade Runner 2049', 'Prisoners']


`yield from page` does exactly what the inner `for` loop did. Use it whenever a generator's job is to pass along the contents of something else.

Python builds the iterator for you. A function containing `yield` is not a normal function at all — calling it runs *none* of the body:

In [21]:
gen = search_results(pages)

print(type(gen))

<class 'generator'>


Print the generator itself and you get something like `<generator object search_results at 0x7f9c1a2b3c40>` — the hex is just where it happens to sit in memory, which is why it differs every time you run it. It tells you nothing about the values, because at this point there are no values.

## What `yield` Actually Does

`return` ends a function permanently. **`yield` pauses it**, hands a value back, and keeps every local variable alive so the function can resume exactly where it stopped.

The clearest way to see this is to put prints around it:

In [22]:
def countdown(n):
    print(f"    [starting at {n}]")
    while n > 0:
        print(f"    [about to yield {n}]")
        yield n
        print(f"    [resumed, now decrementing]")
        n -= 1
    print("    [function finished]")

In [23]:
gen = countdown(3)
print("generator created -- notice nothing above has run yet")

generator created -- notice nothing above has run yet


The function body has not started. Now ask for one value:

In [24]:
print("first value:", next(gen))

    [starting at 3]
    [about to yield 3]
first value: 3


It ran up to the first `yield` and **stopped there**. The function is paused mid-loop, with `n` still equal to 3, waiting.

Ask for another and it picks up on the line after the `yield`:

In [25]:
print("second value:", next(gen))

    [resumed, now decrementing]
    [about to yield 2]
second value: 2


In [26]:
print("third value:", next(gen))

try:
    next(gen)
except StopIteration:
    print("StopIteration -- the function ran off the end")

    [resumed, now decrementing]
    [about to yield 1]
third value: 1
    [resumed, now decrementing]
    [function finished]
StopIteration -- the function ran off the end


So a generator is a function that can be **paused and resumed**, and `StopIteration` is raised automatically when the function finally ends. That is the whole idea.

## What About `return`?

We said a generator uses `yield` *instead of* `return`. That is not quite the whole story: a generator may still contain `return`, and it means **stop now**. It does not hand a value back the way it would in an ordinary function.

In [27]:
def countdown_to_zero(n):
    while True:
        if n == 0:
            return              # ends the generator here; produces no value
        yield n
        n -= 1

print(list(countdown_to_zero(3)))

[3, 2, 1]


So the rule is: **`yield` produces a value, `return` ends the generator.** You will see this used as an early exit, exactly as in the API example at the end of this chapter.

## Getting a Value Without the Exception

Wrapping `next()` in `try`/`except` is noisy when all you want is "the next value, or a fallback". `next()` takes an optional second argument for exactly that:

In [28]:
gen = (x for x in [10, 20])

print(next(gen, "nothing left"))
print(next(gen, "nothing left"))
print(next(gen, "nothing left"))    # exhausted -- returns the default instead of raising

10
20
nothing left


## Generators Are Lazy

Because the body only runs when you ask for a value, a generator can describe a sequence that never ends. This is impossible with a list:

In [29]:
def ticket_numbers(start):
    n = start
    while True:                 # no exit condition at all
        yield n
        n += 1

In [30]:
tickets = ticket_numbers(1001)

for t in tickets:
    print(t, end=" ")
    if t >= 1005:
        break                   # we decide when to stop, not the generator

1001 1002 1003 1004 1005 

An infinite loop that does not hang, because nothing is computed until it is requested. `ticket_numbers` is still sitting there paused, ready to continue if we ask.

This is the practical difference between a list and a generator:

| | List | Generator |
|---|---|---|
| When values are computed | All at once, up front | One at a time, on demand |
| Memory used | Grows with the number of items | Roughly constant |
| Can be infinite | No | Yes |
| Can be reused | Yes | No — one pass only |
| Supports `len()` / indexing | Yes | No |

---
# 4. Generator Expressions

Chapter 5 introduced list comprehension. There is a generator version, and the only difference is the brackets:

In [31]:
squares_list = [x * x for x in range(10)]      # square brackets -> a list
squares_gen  = (x * x for x in range(10))      # parentheses     -> a generator

print(squares_list)
print(type(squares_gen))

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
<class 'generator'>


The list computed ten values immediately. The generator computed none.

With ten items that is irrelevant. With ten million it is the difference between working and crashing:

In [32]:
import sys

big_list = [x * x for x in range(1_000_000)]
big_gen  = (x * x for x in range(1_000_000))

print("list:     ", f"{sys.getsizeof(big_list):>9,} bytes")
print("generator:", f"{sys.getsizeof(big_gen):>9,} bytes")

list:      8,448,728 bytes
generator:       200 bytes


The list holds a million numbers. The generator holds a set of instructions for producing them, so its size does not depend on how many there are — the figure would be identical for a billion.

(Two caveats: the exact byte counts depend on your Python version and platform, and `sys.getsizeof` on a list measures only the list itself, not the objects inside it — so the real gap is even wider than it looks. The file example later in this chapter measures both.)

Both give the same answer:

In [33]:
print(sum(x * x for x in range(1_000_000)))

333332833333500000


Notice there are no extra parentheses in that call. When a generator expression is the only argument to a function you can drop them, which is why you will see `sum(...)`, `max(...)`, `any(...)` and `all(...)` written this way constantly.

**A rule of thumb for choosing:**

- Need the values **more than once**, or need `len()` or indexing? Use a **list**.
- Feeding the values straight into `sum()`, `max()`, a `for` loop, or another pipeline? Use a **generator**.
- Working with something big enough that you would notice the memory? Use a **generator**.

## The Comprehension Family

While we are here: comprehensions come in four forms, and chapter 5 only covered the first. The syntax is identical apart from the brackets.

In [34]:
titles = ["Stranger Things", "The Crown", "Wednesday", "Dark", "The Crown"]

as_list = [t.lower() for t in titles]                 # [ ] -> list, keeps duplicates
as_set  = {t.lower() for t in titles}                 # { } -> set, drops duplicates
as_dict = {t: len(t) for t in titles}                 # {k: v} -> dict
as_gen  = (t.lower() for t in titles)                 # ( ) -> generator

print("list:", as_list)
print("set: ", sorted(as_set))
print("dict:", as_dict)
print("gen: ", type(as_gen).__name__)

list: ['stranger things', 'the crown', 'wednesday', 'dark', 'the crown']
set:  ['dark', 'stranger things', 'the crown', 'wednesday']
dict: {'Stranger Things': 15, 'The Crown': 9, 'Wednesday': 9, 'Dark': 4}
gen:  generator


The **dict** version is the one you will reach for most often after lists. `{key: value for ...}` — the colon inside the braces is what makes it a dict rather than a set:

In [35]:
runtimes = {"Dune": 155, "Arrival": 116, "Sicario": 121, "Prisoners": 153}

# Convert to hours, keeping only the long ones
long_films = {title: round(mins / 60, 1) for title, mins in runtimes.items() if mins > 120}

print(long_films)

{'Dune': 2.6, 'Sicario': 2.0, 'Prisoners': 2.5}


And the **set** version is a neat way to collect distinct values in one line:

In [36]:
watch_log = [
    {"user": "amit", "genre": "Sci-Fi"},
    {"user": "sara", "genre": "Drama"},
    {"user": "amit", "genre": "Sci-Fi"},
    {"user": "raj",  "genre": "Comedy"},
]

genres = {entry["genre"] for entry in watch_log}
print(sorted(genres))

['Comedy', 'Drama', 'Sci-Fi']


| Brackets | Produces | Use when |
|---|---|---|
| `[ ... ]` | list | You need the values more than once |
| `{ ... }` | set | You want unique values |
| `{k: v ...}` | dict | You are building a lookup |
| `( ... )` | generator | You will consume the values once |

---
# 5. Reading Large Files Lazily

This is where it stops being theory. Let us build a file big enough to matter.

In [37]:
import random
import shutil
from pathlib import Path

demo = Path("generators_demo")
if demo.exists():
    shutil.rmtree(demo)
demo.mkdir()

random.seed(7)
artists = ["Arijit Singh", "Taylor Swift", "The Weeknd", "A.R. Rahman", "Drake"]

log = demo / "play_history.csv"
with open(log, "w") as f:
    f.write("artist,ms_played\n")
    for _ in range(200_000):
        f.write(f"{random.choice(artists)},{random.randint(5_000, 240_000)}\n")

print(f"{log} created: {log.stat().st_size / 1_000_000:.1f} MB, 200,000 rows")

generators_demo/play_history.csv created: 3.5 MB, 200,000 rows


**The way that uses the most memory** — read everything, then process it:

In [38]:
with open(log) as f:
    next(f)                             # skip the header
    all_lines = f.readlines()           # every remaining line, in memory, right now

total_bytes = sys.getsizeof(all_lines) + sum(sys.getsizeof(l) for l in all_lines)
print(f"{len(all_lines):,} lines held in memory: ~{total_bytes / 1_000_000:.1f} MB")

200,000 lines held in memory: ~13.3 MB

**The lazy way.** A file object is already an iterator over its lines, so looping over it directly reads one line at a time and never holds the whole file:

In [39]:
with open(log) as f:
    header = next(f)                    # a file supports next() -- it is an iterator
    line_count = sum(1 for _ in f)      # count without storing anything

print(f"counted {line_count:,} lines, holding one at a time")

counted 200,000 lines, holding one at a time

Same answer, and memory use that does not depend on the size of the file. A 50 GB file would work identically.

## Building a Pipeline

Generators become genuinely powerful when you chain them. Each stage takes an iterator and yields another, so nothing is ever fully materialised — data flows through one row at a time.

In [40]:
def read_rows(path):
    # Stage 1: yield each line, skipping the header.
    with open(path) as f:
        next(f)
        for line in f:
            yield line.rstrip("\n")


def parse(rows):
    # Stage 2: turn each line into a (artist, ms) tuple.
    for row in rows:
        artist, ms = row.split(",")
        yield artist, int(ms)


def long_plays(records, minimum_ms):
    # Stage 3: keep only the ones that played for a while.
    for artist, ms in records:
        if ms >= minimum_ms:
            yield artist, ms

Wiring them together reads like a description of the task, and still nothing has been computed:

In [41]:
rows = read_rows(log)
records = parse(rows)
serious_listens = long_plays(records, 180_000)

print(type(serious_listens).__name__)   # a generator -- the file has not been read yet

generator


The work happens only when something consumes it:

In [42]:
from collections import defaultdict

totals = defaultdict(int)
for artist, ms in serious_listens:
    totals[artist] += ms

for artist, ms in sorted(totals.items(), key=lambda pair: pair[1], reverse=True):
    print(f"{artist:<15} {ms / 3_600_000:>6.1f} hours")

Arijit Singh     601.8 hours
Taylor Swift     598.2 hours
A.R. Rahman      596.8 hours
The Weeknd       593.9 hours
Drake            591.0 hours


Two hundred thousand rows aggregated, with **one row in memory at any moment**. Change `200_000` to `200_000_000` and the code does not change — only the runtime does.

This is exactly the shape of a data-cleaning pipeline, and it is why generators appear everywhere in data engineering.

---
# 6. When a Generator Is the Wrong Choice

Generators are not a free upgrade. Three things you give up:

**1. You can only consume it once.** This is the bug that catches everyone:

In [43]:
scores = (n for n in [4.5, 3.8, 4.9, 4.1])

print("average:", sum(scores) / 4)
print("highest:", max(scores, default="nothing left!"))

average: 4.325
highest: nothing left!


The `sum()` consumed the whole generator. By the time `max()` runs there is nothing to look at — and without that `default=` it would have raised a `ValueError` instead.

If you need the values twice, **build a list**:

In [44]:
scores = [n for n in [4.5, 3.8, 4.9, 4.1]]     # a list, so it can be reused

print("average:", sum(scores) / len(scores))
print("highest:", max(scores))

average: 4.325
highest: 4.9


**2. No `len()` and no indexing.** A generator does not know how many values it will produce, and cannot jump to the fifth one:

In [45]:
gen = (x for x in range(10))

try:
    len(gen)
except TypeError as e:
    print("len(gen) ->", e)

try:
    gen[3]
except TypeError as e:
    print("gen[3]   ->", e)

len(gen) -> object of type 'generator' has no len()
gen[3]   -> 'generator' object is not subscriptable


**3. Debugging is less obvious.** Printing a generator shows you `<generator object ...>` and not its contents — and if you call `list()` on it to peek, you have just consumed it and the rest of your code will find it empty.

**But there is a third option**, for when you need to iterate more than once *and* the data is too large to hold as a list. Write a small **iterable** class whose `__iter__` returns a *fresh* generator each time it is called:

In [46]:
class PlayHistory:
    # An iterable, not an iterator: __iter__ hands back a brand new generator
    # on every call, so this object can be looped over as often as you like.
    def __init__(self, path):
        self.path = path

    def __iter__(self):
        with open(self.path) as f:
            next(f)                             # skip the header
            for line in f:
                artist, ms = line.rstrip("\n").split(",")
                yield artist, int(ms)

In [47]:
history = PlayHistory(log)

print(f"rows:         {sum(1 for _ in history):,}")            # one full pass
print(f"longest play: {max(ms for _, ms in history):,} ms")    # another full pass, no problem

rows:         200,000
longest play: 239,999 ms


Two complete passes over the file, still one row in memory at a time, and no exhaustion bug — because each `for` loop gets its own generator. This is how libraries expose large datasets that you can iterate repeatedly.

**So: use a list when** you need the values more than once, need a count, need to index or slice, or the collection is small enough that none of this matters.

**Use a generator when** the data is large, the source is slow, the sequence is endless, or you are building a pipeline that passes values straight through.

## Where You Will Meet Generators

This is not a niche feature you will use occasionally. It is everywhere in the tools you are heading towards:

- **File objects** — `for line in f` is the iterator protocol, as we just saw.
- **`csv.reader`** from chapter 10 yields rows one at a time rather than building a list.
- **`range`, `enumerate`, `zip`, `map`, `filter`** — all lazy, all producing values on demand.
- **pandas** `read_csv(..., chunksize=...)` hands back an iterator of chunks, so you can process a file larger than memory.
- **PyTorch and TensorFlow** data loaders are iterators that yield one batch of training data at a time, for the same reason.

Recognising the pattern is most of the benefit. When a library gives you back something that prints as an object instead of your data, it is almost always an iterator waiting to be consumed.

---
# Real-World Example: Paging Through an API

Here is the pattern that made generators click for a lot of people, and one you will use directly in chapter 17.

An API gives you results one page at a time. You want to write `for user in all_users(...)` and never think about pages again. A generator hides the paging completely:

In [48]:
def fetch_page(page_number):
    # Stands in for a real network call -- chapter 17 replaces this with requests.
    fake_data = {
        1: ["amit", "sara", "raj"],
        2: ["neha", "vikram"],
        3: [],                          # an empty page means we have reached the end
    }
    print(f"    [fetching page {page_number}]")
    return fake_data.get(page_number, [])


def all_users():
    page = 1
    while True:
        users = fetch_page(page)
        if not users:                   # no more results
            return                      # ends the generator
        for user in users:
            yield user
        page += 1

The caller sees a flat list of users, with no idea pages exist:

In [49]:
for user in all_users():
    print(user)

    [fetching page 1]
amit
sara
raj
    [fetching page 2]
neha
vikram
    [fetching page 3]


Look at the order of that output. The fetch messages are **interleaved** with the results: page 2 is not requested until page 1's users have all been handed over. If the caller stops early, the remaining pages are never fetched at all:

In [50]:
for user in all_users():
    print(user)
    if user == "sara":
        print("found who we were looking for, stopping")
        break

    [fetching page 1]
amit
sara
found who we were looking for, stopping


One page fetched instead of three. With a real API that is a saved network request, and on a large dataset it is the difference between a script that starts working immediately and one that downloads for ten minutes first.

---
# Quick Reference

**The protocol**

```python
iter(x)      # get an iterator from an iterable
next(it)     # get the next value; raises StopIteration when empty
```

A `for` loop calls `iter()` once, then `next()` repeatedly, and catches `StopIteration` for you.

**Writing an iterator the long way**

```python
class MyIterator:
    def __iter__(self):
        return self
    def __next__(self):
        if done:
            raise StopIteration
        return value
```

**Writing the same thing as a generator**

```python
def my_generator():
    while not done:
        yield value          # pauses here, resumes on the next request
```

**Comprehensions**

| Syntax | Result |
|---|---|
| `[x for x in xs]` | list |
| `{x for x in xs}` | set |
| `{k: v for k, v in xs}` | dict |
| `(x for x in xs)` | generator |

**Choosing**

| Use a list | Use a generator |
|---|---|
| Values needed more than once | One pass through the data |
| `len()`, indexing or slicing needed | Data too large for memory |
| Small collections | Infinite or unknown-length sequences |
| Easy debugging | Chained processing pipelines |

**The bug to remember**

```python
gen = (x for x in [1, 2, 3])
sum(gen)     # 6
sum(gen)     # 0  -- already exhausted, and nothing warns you
```

---

**Next:** chapter 13 covers closures and decorators — how functions can remember values, and what the `@` symbol you keep seeing actually does.